# A2A Client Workflow — Programmatic Chain Graph

This notebook demonstrates how to build a **programmatic** conversational DAG with
intent-based routing using `mlrun.agentic` chains and MLRun's serving graph.

Unlike the declarative YAML approach (`AgentsAtScaleDeployer`), this example wires
each chain step manually — giving you full control over the graph topology.

### Architecture

```
SessionLoader → RefineQuery → IntentClassifier → IntentChoice
                                                      ├→ A2AClient  (intent="meeting")
                                                      └→ Communicator (else)
                                                           ↓
LanguageGuardrail → HallucinationGuardrail → HistorySaver → respond()
```

- **IntentClassifier** detects whether the query is about a "meeting" or general conversation
- **A2AClient** sends meeting transcripts to an external A2A agent for summarization
- **Communicator** handles general chat via an LLM
- **Guardrails** validate language & hallucination before saving to session history

## 1. Install Dependencies

In [ ]:
!pip install git+https://github.com/mlrun/mlrun@agentic-ai-declerative \
    a2a-sdk==0.3.0 httpx langchain langchain-openai openai python-dotenv

## 2. Setup — Project & Secrets

In [ ]:
import mlrun

PROJECT_NAME = "a2aclient-demo"
project = mlrun.get_or_create_project(PROJECT_NAME)

In [ ]:
# Set your secrets here — these will be injected into the serving function at deploy time.
# A2A_BASE_URL is the URL of the external A2A agent server that handles meeting summaries.
secrets = {
    "OPENAI_BASE_URL": "<your-openai-base-url>",
    "OPENAI_API_KEY": "<your-openai-api-key>",
    "A2A_BASE_URL": "http://localhost:10000",  # URL of the A2A agent server
}
project.set_secrets(secrets=secrets)

import os
os.environ["OPENAI_BASE_URL"] = secrets["OPENAI_BASE_URL"]
os.environ["OPENAI_API_KEY"] = secrets["OPENAI_API_KEY"]
os.environ["A2A_BASE_URL"] = secrets["A2A_BASE_URL"]

## 3. Write Source File

The `graph_initializer` function runs once when the serving graph starts up.
It sets the artifact-backed `SessionStore` on the server context so that
`SessionLoader` and `HistorySaver` can read/write session history.

In [ ]:
%%writefile source.py
import mlrun
import mlrun.serving as mlrun_serving
from mlrun.agentic.sessions import SessionStore

PROJECT_NAME = "a2aclient-demo"


def graph_initializer(server: mlrun_serving.GraphServer):
    """Called once when the serving graph starts up."""
    context = server.context
    if getattr(context, "session_store", None) is None:
        project = mlrun.get_or_create_project(PROJECT_NAME)
        context.session_store = SessionStore(project=project)

## 4. Build Serving Function & Wire the Graph

This is the core of the programmatic approach: we create chain instances,
then wire them into an MLRun serving DAG with `.to()` and `.add_step()`.

In [ ]:
import os
from mlrun.agentic.chains.a2a_client import A2AClient
from mlrun.agentic.chains.base import HistorySaver, SessionLoader
from mlrun.agentic.chains.communicator import Communicator
from mlrun.agentic.chains.hallucination_guardrail import HallucinationGuardrail
from mlrun.agentic.chains.intent_choice import IntentChoice
from mlrun.agentic.chains.intent_classifier import IntentClassifier
from mlrun.agentic.chains.language_guardrail import LanguageGuardrail
from mlrun.agentic.chains.refine import CONVERSATION_CONTEXT_REFINER_PROMPT, RefineQuery

FUNCTION_NAME = "a2a-workflow"

serving_fn = project.set_function(
    name=FUNCTION_NAME,
    func="source.py",
    kind="serving",
    image="mlrun/mlrun",
    requirements=[
        "a2a-sdk==0.3.0",
        "httpx>=0.24,<1.0",
        "langchain",
        "langchain-openai",
        "openai",
        "python-dotenv",
    ],
)
serving_fn.spec.graph_initializer = "graph_initializer"

# -- Build the DAG --
root = serving_fn.set_topology("flow", engine="async")

# Chain instances
session_loader = SessionLoader(name="session-loader")
refine_query = RefineQuery(
    name="refine-query",
    prompt_template=CONVERSATION_CONTEXT_REFINER_PROMPT,
)
intent_classifier = IntentClassifier(name="intent-classifier")
intent_choice = IntentChoice(name="intent-choice")
a2a_client = A2AClient(
    name="Atomic Agent(a2a)",
    base_url=os.getenv("A2A_BASE_URL", "http://localhost:10000"),
)
communicator = Communicator(name="communicator")
language_guardrail = LanguageGuardrail(name="language-guardrail")
hallucination_guardrail = HallucinationGuardrail(name="hallucination-guardrail")
history_saver = HistorySaver(name="history-saver")

# First part: session → refine → classify → choice
classify_task = root.to(session_loader).to(refine_query).to(intent_classifier)
choice_task = classify_task.to(intent_choice)

# Choice branches
choice_task.to(a2a_client)
choice_task.to(communicator)

# Merge branches back, then guardrails → save → respond
language_guardrail_task = root.add_step(
    language_guardrail, after=["Atomic Agent(a2a)", "communicator"]
)
language_guardrail_task.to(hallucination_guardrail).to(history_saver).respond()

print("Serving function built successfully.")

## 5. Visualize the Graph

In [ ]:
serving_fn.spec.graph.plot(rankdir="LR")

## 6. Test Locally with Mock Server

The mock server runs the full graph in-process — no cluster deployment needed.
This is useful for iterating on the graph logic before deploying.

In [ ]:
mock_server = serving_fn.to_mock_server()

In [ ]:
# Test with a casual query (goes through Communicator branch)
resp = mock_server.test("", body={"query": "What is the capital of France?"})
print(resp)

In [ ]:
# Test with a meeting transcript (goes through A2AClient branch)
# Note: requires the A2A agent server to be running at A2A_BASE_URL
meeting_transcript = """
Alice: Let's start. The deadline for the API migration is next Friday.
Bob: I can handle the auth endpoints. Should be done by Wednesday.
Alice: Great. Charlie, can you update the docs?
Charlie: Sure, I'll have them ready by Thursday.
Alice: Perfect. Let's sync again on Wednesday afternoon.
"""

resp = mock_server.test("", body={"query": f"Summarize this meeting transcript: {meeting_transcript}"})
print(resp)

## 7. Deploy to Iguazio Cluster

Deploy the serving function to the cluster. The secrets set in step 2 will
be automatically injected as environment variables.

In [ ]:
serving_fn.deploy()

## 8. Test the Deployed Function

In [ ]:
# Casual query
resp = serving_fn.invoke("", body={"query": "Hello, how are you?"})
print(resp)

In [ ]:
# Meeting summary query (with session tracking)
resp = serving_fn.invoke("", body={
    "query": f"Summarize this meeting transcript: {meeting_transcript}",
    "username": "demo-user",
    "session_name": "my-session",
})
print(resp)

---

## Architecture Notes

### Programmatic vs Declarative

| | Programmatic (this notebook) | Declarative (teams-demo) |
|---|---|---|
| **Graph definition** | Python code (`.to()`, `.add_step()`) | YAML + `AgentsAtScaleDeployer` |
| **Routing** | Custom `IntentChoice` (storey.Choice) | Strategy-based (sequential, selector, etc.) |
| **Use case** | Custom DAGs with arbitrary logic | Standard multi-agent patterns |
| **Session management** | Same `SessionStore` | Same `SessionStore` |

### Key Components

- **`SessionStore(project=...)`** — stores session history as MLRun artifacts
- **`SessionLoader` / `HistorySaver`** — read/write session state at graph entry/exit
- **`RefineQuery`** — rewrites the user query using conversation context
- **`IntentClassifier`** — classifies intent (e.g., "meeting" vs general)
- **`IntentChoice`** — routes to different branches based on classified intent
- **`A2AClient`** — calls an external A2A agent via the A2A protocol
- **`Communicator`** — general LLM chat
- **`LanguageGuardrail` / `HallucinationGuardrail`** — output validation